## Time history diagnostics

In [ ]:
# Import needed modules
import numpy as np
import matplotlib.pylab as plt
import os

In [ ]:
# Set working directory
sim_name = "Landau"
tail = "_zigzag"
pathname = '/u/sonnen/gempic_gpu_runs/'+sim_name + tail
pathname_out = pathname + '/processed'
try:
    os.mkdir(pathname_out)
except(FileExistsError):
    
    pass
os.chdir(pathname)

In [ ]:
# Read time history output file
file = open(sim_name + '.txt', 'r')
lines = file.readlines()
nsteps = len(lines)-1  # First line is not counted
print("nsteps=",nsteps)
time = np.zeros(nsteps)
ex = np.zeros(nsteps)
ey = np.zeros(nsteps)
ez = np.zeros(nsteps)
bx = np.zeros(nsteps)
by = np.zeros(nsteps)
bz = np.zeros(nsteps)
ekin = np.zeros(nsteps)
px = np.zeros(nsteps)
py = np.zeros(nsteps)
pz = np.zeros(nsteps)
errgauss = np.zeros(nsteps)
for i in range(nsteps):
    vals = lines[i+1].rstrip().split(' ')
    time[i] = vals[0]
    ex[i] = vals[1]
    ey[i] = vals[2]
    ez[i] = vals[3]
    bx[i] = vals[4]
    by[i] = vals[5]
    bz[i] = vals[6]
    ekin[i] = vals[7]
    px[i] = vals[8]
    py[i] = vals[9]
    pz[i] = vals[10]
    errgauss[i] = vals[11]
    #print(i,vals)
print(min(ex),max(ex))
# Close opened file
file.close()

In [ ]:
vol= 4*np.pi
fig, axs = plt.subplots(2, 2,sharex=True,tight_layout=True)
axs[0,0].plot(time,ex+ey+ez)
axs[0,0].set_title('electric energy')
axs[1,0].plot(time,bx+by+bz)
axs[1,0].set_title('magnetic energy')
axs[0,1].plot(time,ekin)
axs[0,1].set_title('kinetic energy')
axs[1,1].plot(time,0.5*(ex+ey+ez+bx+by+bz)*vol+ekin)
axs[1,1].set_title('total energy');

In [ ]:
# Plot |Ex|**2 
fig, ax = plt.subplots()
ax.semilogy(time, ex, 'k')
ax.set_xlabel('time')
ax.set_ylabel('$Ex^2$')
ax.set_title('$Ex^2$ vs. Time')
ax.legend([sim_name],loc='upper right')
plt.savefig(pathname_out + '/E1_squared_vs_t.jpg');

In [ ]:
# Plot |Ex|**2  vs. time
fig, ax = plt.subplots()
ax.semilogy(time, bz, 'k')
ax.semilogy(time, ex, 'b')
ax.semilogy(time, ey, 'r')
ax.semilogy(time,(0.05*np.pi*0.4246*np.exp(-0.0661*time)*np.cos(1.2850*time-0.335))**2)
#ax.semilogy(time,4.e-3*np.exp(-2*0.066*time))
ax.set_xlabel('time')
ax.set_ylabel('$Ex^2$')
ax.set_title('$Ex^2$ vs. Time')
ax.legend([sim_name],loc='upper right')
plt.savefig(pathname_out + '/E1_squared_vs_t.jpg');

In [ ]:
from scipy.signal import find_peaks

i1 = 50
i2 = 1200
plt.semilogy(ex[i1:i2])
#plt.semilogy((0.1*0.4246*np.exp(-0.0661*time[i1:i2])*np.cos(1.2850*time[i1:i2]-0.335))**2)
plt.semilogy((0.05*np.pi*0.4246*np.exp(-0.0661*time[i1:i2])*np.cos(1.2850*time[i1:i2]-0.335))**2)

imax, Eval_dict = find_peaks(ex[i1:i2],height=0)
Eval = Eval_dict['peak_heights']
slope = np.zeros(Eval.shape)
freq = np.zeros(Eval.shape)
for i in range(Eval.size-1):
    slope[i] = (np.log(Eval[i+1])-np.log(Eval[i]))/(time[imax[i+1]]-time[imax[i]])
    freq[i] = np.pi/(time[imax[i+1]] - time[imax[i]])
slope_av = np.average(slope[1:-1])
freq_av = np.average(freq[1:-1])
print("damping rate", slope_av/2)
print("frequency", freq_av)
print("damping rate", slope/2)
print("frequency", freq)

In [ ]:
# Plot Gauss Law Error 
fig, ax = plt.subplots()
ax.plot(time, errgauss);
ax.set_xlabel('time');
ax.set_ylabel('Gauss Law Error');
ax.set_title('Gauss Law Error vs. Time');
ax.legend([sim_name],loc='upper left')
plt.savefig(pathname_out + '/Gauss_Law_Error_vs_t.jpg');